# Bronze — contract-validated, retry-safe file batch

`replaceWhere` atomically replaces only the selected source_key partitions of logical data. This is a bounded batch, not a full historical rewrite. Review the table column layout before first run.

In [0]:
from datetime import datetime, timezone
from uuid import uuid4

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType

# ============================================================
# 1. CONFIGURATION
# ============================================================

CATALOG = "nyc_citi_bike_catalog"

FILES_TABLE = f"{CATALOG}.audit.audit_files_tbl"
EVENTS_TABLE = f"{CATALOG}.audit.audit_events_tbl"
BRONZE_TABLE = f"{CATALOG}.bronze.raw_trips_tbl"

# Start with True to inspect pending files without writing.
DRY_RUN = True

TRIP_COLUMNS = [
    "ride_id",
    "rideable_type",
    "started_at",
    "ended_at",
    "start_station_name",
    "start_station_id",
    "end_station_name",
    "end_station_id",
    "start_lat",
    "start_lng",
    "end_lat",
    "end_lng",
    "member_casual",
]

TRIP_SCHEMA = StructType([
    StructField(name, StringType(), True)
    for name in TRIP_COLUMNS
])

# ============================================================
# 2. READ EXISTING AUDIT TABLES
# ============================================================

files_df = spark.table(FILES_TABLE)
events_df = spark.table(EVENTS_TABLE)

In [0]:
# Successfully extracted CSVs
extracted_df = (
    events_df
    .filter(
        (F.col("stage") == "CSV_EXTRACT") &
        (F.col("status") == "SUCCESS")
    )
    .select("source_key")
    .where(F.col("source_key").isNotNull())
    .distinct()
)

# Successfully loaded into Bronze
completed_df = (
    events_df
    .filter(
        (F.col("stage") == "BRONZE") &
        (F.col("status") == "SUCCESS")
    )
    .select("source_key")
    .where(F.col("source_key").isNotNull())
    .distinct()
)

# ============================================================
# 3. IDENTIFY PENDING FILES (DISTRIBUTED)
# ============================================================

pending_keys_df = extracted_df.join(
    completed_df,
    on="source_key",
    how="left_anti"
)

In [0]:
# Select the latest extraction record for each source key.
# This prevents duplicate file metadata from duplicating rows
# when joining lineage to the CSV contents.

window = (
    Window.partitionBy("source_key")
    .orderBy(
        F.col("extracted_at").desc_nulls_last(),
        F.col("file_id").desc()
    )
)

pending_files_df = (
    files_df
    .join(pending_keys_df, "source_key", "inner")
    .withColumn("_rn", F.row_number().over(window))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
    .select(
        "file_id",
        "source_key",
        "zip_key",
        "batch_id",
        "csv_path",
        "csv_name",
        "source_zip_name",
        "source_url",
        "data_month",
        "file_size_bytes"
    )
)


In [0]:
pending_files_df.display()

In [0]:
from pyspark.sql import functions as F

# 1. Get pending CSV paths
paths = [
    r.csv_path
    for r in pending_files_df.select("csv_path").distinct().collect()
]

if paths:
    # 2. Read CSVs in parallel using Spark
    raw_df = (
        spark.read
        .format("csv")
        .option("header", True)
        .option("mode", "PERMISSIVE")
        .schema(TRIP_SCHEMA)
        .load(paths)
        .withColumn(
            "csv_path",
            F.regexp_replace(F.col("_metadata.file_path"), "^dbfs:", "")
        )
    )

    # 3. Attach source metadata
    bronze_df = (
        raw_df.join(
            pending_files_df.select(
                "csv_path", "source_key", "batch_id"
            ).dropDuplicates(["csv_path"]),
            on="csv_path",
            how="inner"
        )
        .drop("csv_path")
    )

    # 4. Append to Bronze
    (
        bronze_df.write
        .format("delta")
        .mode("append")
        .saveAsTable("nyc_citi_bike_catalog.bronze.raw_trips_tbl")
    )

    print("Bronze ingestion completed")
else:
    print("No pending files")

In [0]:
bronze_df.display()

In [0]:
row_count_df = bronze_df.groupBy(F.col("source_key"), F.col("batch_id")).agg(F.count("ride_id").alias('bronze_row_written'))
row_count_df.display()

In [0]:
now = datetime.now(timezone.utc).isoformat()

event_records = []

for record in pending_files_df.collect():
    event_records.append(
        {
            "event_id": str(uuid4()),
            "batch_id": record.batch_id,
            "zip_key": record.zip_key,
            "source_key": record.source_key,
            "stage": "BRONZE",
            "status": "SUCCESS",
            "source_zip_name": record.source_zip_name,
            "source_url": record.source_url,
            "data_month": record.data_month,
            "target_path": BRONZE_TABLE,
            "size_bytes": record.file_size_bytes,
            "file_count": 1,
            "started_at": now,
            "finished_at": now,
            "error_message": None,
        }
    )

audit_schema = spark.table(EVENTS_TABLE).schema

audit_df = spark.createDataFrame(event_records, schema=audit_schema)

audit_df = (
    audit_df.alias("audit")
    .join(
        row_count_df.select(
            F.col("source_key").alias("bronze_source_key"), F.col("bronze_row_written")
        ).alias("bronze_counts"),
        on=F.col("audit.source_key") == F.col("bronze_counts.bronze_source_key"),
        how="inner",
    )
    .withColumn("rows_written", F.col("bronze_row_written"))
    .drop("batch_id", "bronze_row_written", "bronze_source_key")
)

audit_df.display()

In [0]:
audit_df.write.format('delta').mode("append").saveAsTable(EVENTS_TABLE)